# 03 — Council, judge and critic

Pre-registered council ablation, conditions A–F, evaluation split n = 71. Labels used in every notebook: **MEASURED** (a real provider or real run produced the number), **ESTIMATED** (derived here from measured data by a stated formula), **SIMULATED** (mock runner, no real model), **NOT_RUN**, **UNKNOWN** (the evidence does not say). Implementation and wiring are reported separately from evaluation: code that exists is not a measured capability.

In [1]:
import json
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 11  # the pre-registered bootstrap seed; reused for every derived computation here
random.seed(SEED)
np.random.seed(SEED)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "EVIDENCE_MAP.md").exists())  # research/council-ablation
EV = ROOT / "evidence"
JEV = ROOT.parent / "jev" / "evidence"  # the JEV study's canonical copy in this repository
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)


def load(rel):
    return json.loads((EV / rel).read_text())


def jsonl(rel):
    return [json.loads(x) for x in (EV / rel).read_text().splitlines() if x.strip()]


def final_calls(rel):
    """The harness cache is append-only: a re-asked call (Amendments 1 and 2) appends a new row and the
    last row per (member, case) is the one analysed. Earlier rows stay in the file as recorded calls."""
    return pd.DataFrame(jsonl(rel)).drop_duplicates(["member", "case_id"], keep="last").reset_index(drop=True)


R = load("council/results.json")
REG = {r["id"]: r for r in load("reasoning_methods/reasoning_methods_registry.json")}
print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__, "| seed", SEED)
print("evidence files:", sorted(str(p.relative_to(ROOT)) for p in EV.rglob("*") if p.is_file()))


python 3.12.12 | pandas 3.1.0 | numpy 2.5.4 | seed 11
evidence files: ['evidence/SOURCE_MANIFEST.json', 'evidence/council/calibration_by_member.csv', 'evidence/council/cases.csv', 'evidence/council/cases.json', 'evidence/council/conditions.csv', 'evidence/council/cost_latency_by_condition.csv', 'evidence/council/judge_calls.csv', 'evidence/council/judge_calls.jsonl', 'evidence/council/member_calls.csv', 'evidence/council/member_calls.jsonl', 'evidence/council/member_calls_budget300_archive.csv', 'evidence/council/member_calls_budget300_archive.jsonl', 'evidence/council/results.json', 'evidence/decision_methods/ablation-2026-10-20.json', 'evidence/notebook_runs.json', 'evidence/publication_scan.json', 'evidence/reasoning_methods/method_status_summary.csv', 'evidence/reasoning_methods/reasoning_methods_registry.json', 'evidence/reasoning_methods/reasoning_strategies_mock_benchmark.md', 'evidence/reasoning_methods/research_plans.json', 'evidence/replay_check.json']


In [2]:
#: Evaluation state per method, with the evidence that sets it. Explicit, not inferred from text.
EVALUATION = {
    "deterministic-baseline": "MEASURED",      # reference arm of the decision-methods ablation
    "single-strong-model": "MEASURED",         # condition A of the council benchmark
    "council": "MEASURED",                     # conditions B1-D, F (harness, not the product endpoint)
    "proposer-judge": "MEASURED",              # as condition E (judge over C) in the harness only
    "critic-veto": "NOT_RUN",
    "critic-selective-council": "NOT_RUN",
    "cot": "SIMULATED (mock runner)",
    "tot": "SIMULATED (mock runner)",
    "got": "SIMULATED (mock runner)",
    "self-consistency": "NOT_RUN",
    "react": "NOT_RUN",
    "reflexion": "NOT_RUN",
    "pot-evidence": "NOT_RUN",
    "mcts": "NOT_RUN",
    "jev": "MEASURED",                         # real provider calls on decision-corpus-v1
    "agentic-llm-decision": "MEASURED",        # decision-methods ablation, small local model
    "hermes-agent-patterns": "NOT_RUN",
}


def status_row(method_id):
    """Implementation / wiring / evaluation labels, derived from the registry snapshot only."""
    r = REG[method_id]
    implemented = "NOT_IMPLEMENTED" if r["wiring"] == "NOT_PRESENT" else "IMPLEMENTED"
    wired = "WIRED" if r["wiring"] in {"WIRED", "DECISION_AFFECTING", "OBSERVED", "EVALUATED"} else "NOT_WIRED"
    evaluation = EVALUATION[method_id]
    return {"method": r["name"], "implemented": implemented, "wired": wired, "authority": r["authority"],
            "evaluation": evaluation, "production": r["production"], "recommendation": r["recommendation"],
            "result": r["result"], "limitations": r["limitations"]}


In [3]:
pd.DataFrame([status_row(m) for m in ("council", "proposer-judge", "critic-veto", "critic-selective-council")]).T

,0,1,2,3
method,Council (multi-persona / multi-model vote),Proposer / Judge (training champion re-evaluation),Semantic critic with protective-minority veto,Critic + selective Council
implemented,IMPLEMENTED,IMPLEMENTED,NOT_IMPLEMENTED,NOT_IMPLEMENTED
wired,WIRED,WIRED,NOT_WIRED,NOT_WIRED
authority,ADVISORY,DECIDES,NONE,NONE
evaluation,MEASURED,MEASURED,NOT_RUN,NOT_RUN
production,IN_PRODUCT,IN_PRODUCT_OFF_BY_DEFAULT,NOT_IN_PRODUCT,NOT_IN_PRODUCT
recommendation,SELECTIVE,SELECTIVE,RESEARCH,RESEARCH
result,Pre-registered 110 cases (71 scored): best single model 70/71; no council condition beat it (all six REJECTED_ON_THIS_BENCHMARK); some v...,Judge 0.9859 — tied with council C and the single model; no headroom.,Not tested; the RESULTS.md observation motivates it.,UNKNOWN
limitations,"Ceiling effect (70/71); product endpoint, Validator and tenant context were not part of the benchmark; any claim beyond this set is UNKN...",The judge sees the same validation-split metrics the deterministic pick used; no locked test. Its override authority is not gated by ris...,Hypothesis only.,Not built.


In [4]:
order = ["A_solo:sonnet", "A_solo:haiku", "B1_same_provider_roles", "B2_same_provider_sampling", "C_multi_provider",
         "D_characteristic_diverse", "E_judge", "F_adaptive_routing"]
rows = []
for k in order:
    v = R["conditions"][k]
    rows.append({"condition": k, "members": ", ".join(v["members"]), "accuracy": v["accuracy"],
                 "delta_vs_solo": (v.get("vs_strong_solo") or {}).get("delta"), "ci95": (v.get("vs_strong_solo") or {}).get("ci95"),
                 "critical_miss": v["critical_miss_rate"], "false_alarm": v["false_alarm_rate"],
                 "decision": R["decisions"].get(k, "baseline" if k == "A_solo:sonnet" else "—")})
pd.DataFrame(rows).set_index("condition")

,members,accuracy,delta_vs_solo,ci95,critical_miss,false_alarm,decision
condition,,,,,,,
A_solo:sonnet,sonnet,0.9859,0.0000,"[0.0, 0.0]",0.018519,0.0,baseline
A_solo:haiku,haiku,0.9859,0.0000,"[0.0, 0.0]",0.018519,0.0,—
B1_same_provider_roles,"haiku_verifier, haiku_domain, haiku_critic",0.9718,-0.0141,"[-0.0423, 0.0]",0.018519,0.0,REJECTED
B2_same_provider_sampling,"haiku_s1, haiku_s2, haiku_s3",0.9718,-0.0141,"[-0.0423, 0.0]",0.018519,0.0,REJECTED
C_multi_provider,"sonnet, haiku, qwen7b",0.9859,0.0000,"[0.0, 0.0]",0.018519,0.0,REJECTED
D_characteristic_diverse,"sonnet, haiku_verifier, qwen7b",0.9718,-0.0141,"[-0.0423, 0.0]",0.018519,0.0,REJECTED
E_judge,"sonnet, haiku, qwen7b, judge(sonnet)",0.9859,0.0000,"[0.0, 0.0]",0.018519,0.0,REJECTED
F_adaptive_routing,route,0.9718,-0.0141,"[-0.0423, 0.0]",0.018519,0.0,REJECTED


In [5]:
pd.Series({k: v for k, v in R["hypotheses"].items() if k in ("H0", "H1")}, name="hypothesis").to_frame()

,hypothesis
H0,NOT_REJECTED
H1,NOT_SUPPORTED


**RESULT (MEASURED).** The strong solo model answered 70 of 71 evaluation cases. C and E tied it; B1, B2, D and F scored 69/71. Every council condition is `REJECTED_ON_THIS_BENCHMARK`. H0 is not rejected; H1 is not supported.

**Scope.** With 70/71 for the solo model, the largest possible council gain is +0.014, so the rejection rule is met by construction. The result says nothing about questions a frontier solo model gets wrong. It does **not** show that councils never work.

## Member characteristics (behaviour, not provider names)

In [6]:
mc = pd.DataFrame(R["member_characteristics"]).T
mc[["model", "role", "accuracy", "permissiveness_critical_miss", "conservatism_false_alarm", "ambiguity_handling", "format_failures"]]

,model,role,accuracy,permissiveness_critical_miss,conservatism_false_alarm,ambiguity_handling,format_failures
sonnet,anthropic/claude-sonnet-5-5,neutral,0.9859,0.018519,0.0,0.888889,0
haiku,anthropic/claude-haiku-5-5,neutral,0.9859,0.018519,0.0,0.888889,0
haiku_repeat,anthropic/claude-haiku-5-5,neutral,0.9859,0.018519,0.0,0.888889,0
haiku_s1,anthropic/claude-haiku-5-5,neutral,0.9718,0.018519,0.0,0.777778,0
haiku_s2,anthropic/claude-haiku-5-5,neutral,0.9859,0.018519,0.0,0.888889,0
haiku_s3,anthropic/claude-haiku-5-5,neutral,0.9577,0.018519,0.0,0.666667,0
haiku_verifier,anthropic/claude-haiku-5-5,verifier,0.9577,0.018519,0.058824,0.777778,0
haiku_domain,anthropic/claude-haiku-5-5,domain,0.9718,0.018519,0.0,0.777778,0
haiku_critic,anthropic/claude-haiku-5-5,critic,0.9859,0.0,0.0,1.0,0
qwen7b,ollama/qwen2.5:7b,neutral,0.6197,0.314815,0.235294,0.555556,2


In [7]:
phi = pd.Series(R["pairwise_error_phi_evaluation"], name="error_phi").sort_values()
phi.to_frame()

,error_phi
haiku_verifier~qwen1_5b,-0.335410
haiku_domain~qwen1_5b,-0.271870
haiku~qwen1_5b,-0.190863
sonnet~qwen1_5b,-0.190863
haiku_critic~qwen7b,-0.093628
qwen7b~qwen1_5b,-0.025436
haiku_verifier~haiku_critic,-0.025105
haiku_domain~haiku_critic,-0.020349
haiku~haiku_critic,-0.014286
sonnet~haiku_critic,-0.014286


## Diversity (H2) — descriptive only

In [8]:
H2 = R["hypotheses"]["H2"]
pd.Series({k: v for k, v in H2.items() if not isinstance(v, (list, dict))}).to_frame("H2")

,H2
combinations,35
spearman_gain_vs_error_phi,0.069
spearman_gain_vs_double_fault,-0.83
spearman_gain_vs_provider_count,-0.179
note,Descriptive: combinations overlap and are not independent samples.


In [9]:
pd.DataFrame(H2["mean_gain_by_provider_count"]).T

,n,mean_gain
1,10.0,-0.0042
2,25.0,-0.0563


**INTERPRETATION.** Gain tracks how often members fail on the *same* case (Spearman −0.83 with the double-fault rate) far more than provider count (−0.18). The second provider available was a much weaker local model, so provider count is confounded with member strength. The 35 combinations overlap and are not independent samples.

## The critic was right and outvoted (EXPLORATORY)

In [10]:
calls = final_calls("council/member_calls.jsonl")
truth = pd.DataFrame(load("council/cases.json")["cases"]).set_index("case_id")
b1 = calls[calls.member.isin(["haiku_verifier", "haiku_domain", "haiku_critic"])].pivot(index="case_id", columns="member", values="answer")
b1["truth"] = truth.answer
b1["split"] = truth.split
crit_right = b1[(b1.haiku_critic == b1.truth) & ((b1.haiku_verifier != b1.truth) & (b1.haiku_domain != b1.truth))]
crit_right

member,haiku_critic,haiku_domain,haiku_verifier,truth,split
case_id,,,,,
feature_screening:29,REVIEW,OK,OK,REVIEW,evaluation
feature_screening:34,REVIEW,LEAKAGE,LEAKAGE,REVIEW,evaluation


On these cases plurality voting discarded the one correct dissent. A *protective-minority veto* is a hypothesis for a new pre-registration; it is `NOT_IMPLEMENTED` and `NOT_RUN`.

In [11]:
print("reproducibility:", R["system_quality"]["reproducibility_haiku_T1_two_samples"])
print("validator agreement:", R["system_quality"]["validator_agreement"])
print("organization context:", R["system_quality"]["organization_context_use"])

reproducibility: 110 / 110 identical answers on re-run
validator agreement: NOT_MEASURED — harness calls models directly, not through the product's Validator path
organization context: NOT_MEASURED — cases carry their own facts; no tenant context was injected
